# test-F — C3: Docling chunking + VLM figure descriptions

answering: **gpt-5.6-luna**, figure descriptions: **gpt-5.6-luna**, embedding: **text-embedding-3-small**.

Means over 3 repeats of 150 in-scope and 20 out-of-scope questions. Reads `answers-r*.json` and `judge-r*.json` in this folder; nothing here calls an LLM.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "code"))  # test/code
import imka_eval as E

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RUN = "test-F"
rows = E.run_rows(RUN)
in_scope = rows[rows.group != "out_of_scope"]

## Documents

The five documents the run searched (pinned in `documents.json`).

In [2]:
docs = E._read_json(E.run_dir(RUN) / "documents.json")["documents"]
pd.DataFrame([{"document": d["document_name"][:45], "document_id": d["document_id"], "chunks": d.get("chunk_count"), "captioning": (d.get("models") or {}).get("captioning", "-"), "embedding": (d.get("embedding") or {}).get("model", E.DEFAULT_EMBEDDING)} for d in docs])

,document,document_id,chunks,captioning,embedding
0,ABB Manual for Induction Motors and Generator,41ec3469-76ef-43af-af26-ddc71f09336b,478,gpt-5.6-luna,text-embedding-3-small
1,EMSA Diesel Generator manual-en.pdf,56d1e6e9-8486-4f9f-8ebd-82d82e758c31,285,gpt-5.6-luna,text-embedding-3-small
2,Kohler Generator Manual.pdf,44763515-c04e-4c75-99ea-8eedffd37e93,294,gpt-5.6-luna,text-embedding-3-small
3,Nidec Motor Manual.pdf,218acdfd-52b7-4606-bce8-ee468882aa3b,116,gpt-5.6-luna,text-embedding-3-small
4,Siemens_Manual_Vertical_Induction_Motors_EN.p,a03aca4e-dbdc-4868-a843-4c240457390e,119,gpt-5.6-luna,text-embedding-3-small


## Results by question group

In [3]:
E.summary(runs=(RUN,), by="group")

n  hit@1  hit@3  hit@5  reciprocal_rank  correct_pdf@5  correct_pdf@1  candidates_hit  faithfulness  answer_relevancy  context_recall  answer_accuracy  refused
run    group                                                                                                                                                                           
test-F tables          30  0.733  0.967  0.978            0.847          0.978          0.956           1.000         0.880             0.873           0.922            0.911      NaN
       cross_document  30  0.756  0.922  0.956            0.838          1.000          0.800           1.000         0.894             0.857           0.958            0.914      NaN
       topical         30  0.744  0.889  0.933            0.817          0.967          0.922           0.978         0.937             0.797           0.864            0.872      NaN
       paraphrase      35  0.619  0.848  0.933            0.742          0.990          0.895           0.971         0.886             0.820           0.862            0.833      NaN
       figures         25  0.840  0.880  0.920            0.866          1.000          1.000           0.933         0.860             0.660           0.770            0.727      NaN
       out_of_scope    20    NaN    NaN    NaN              NaN            NaN            NaN             NaN           NaN               NaN             NaN              NaN      1.0

In [4]:
E.summary(runs=(RUN,), by=None)

,n,hit@1,hit@3,hit@5,reciprocal_rank,correct_pdf@5,correct_pdf@1,candidates_hit,faithfulness,answer_relevancy,context_recall,answer_accuracy,refused
run,,,,,,,,,,,,,
test-F,170,0.731,0.9,0.944,0.818,0.987,0.911,0.978,0.892,0.807,0.878,0.855,1.0


## Stability across the three repeats

In [5]:
in_scope.groupby("repeat")[["hit@5", "context_recall", "answer_accuracy"]].mean().round(3)

,hit@5,context_recall,answer_accuracy
repeat,,,
1,0.940,0.888,0.857
2,0.947,0.866,0.840
3,0.947,0.881,0.868


## Out-of-scope questions

Share of the 20 out-of-scope questions declined, per repeat.

In [6]:
rows[rows.group == "out_of_scope"].groupby("repeat")["refused"].mean().round(3)

repeat
1    1.0
2    1.0
3    1.0
Name: refused, dtype: float64

## Lowest answer accuracy

The ten in-scope questions with the lowest accuracy, averaged over the repeats, with the first repeat's answer.

In [7]:
answers = {r["question"]: r.get("answer") or "" for r in E.load_answers(RUN, 1)["results"]}
worst = (in_scope.groupby(["question", "group", "document_name"])["answer_accuracy"].mean()
         .sort_values().head(10).reset_index())
worst["document_name"] = worst["document_name"].str[:20]
worst["answer (repeat 1)"] = worst["question"].map(lambda q: answers.get(q, "")[:160])
worst

,question,group,document_name,answer_accuracy,answer (repeat 1)
0,"As a thumb rule, how often does ABB say an ins...",topical,ABB Manual for Induc,0.0,The provided ABB context does **not state a th...
1,How do I work out equivalent running hours for...,paraphrase,ABB Manual for Induc,0.0,## Equivalent running hours\n\nThe provided co...
2,"In the ABB induction machine manual, what is t...",tables,ABB Manual for Induc,0.0,## Answer\n\nThe provided excerpt does not lis...
3,"What run, alarm and shutdown temperatures does...",cross_document,Nidec Motor Manual.p,0.0,The provided context does not explicitly list ...
4,What is the correct charge generator belt tens...,figures,EMSA Diesel Generato,0.0,## Answer\n\nThe provided context does **not s...
5,What symptoms suggest a faulty temperature sen...,paraphrase,ABB Manual for Induc,0.0,## Indications of a faulty temperature sensor\...
6,What do the parts of the EMSA model code E BD ...,figures,EMSA Diesel Generato,0.0,## Answer\n\nThe context does not provide the ...
7,What does Figure 27 in the Kohler manual show?,figures,Kohler Generator Man,0.0,The provided context does not include Figure 2...
8,"Which engine, lubrication capacity and coolant...",tables,EMSA Diesel Generato,0.0,The provided context does not include an entry...
9,When adding electrolyte to an EMSA maintenance...,figures,EMSA Diesel Generato,0.0,## Required values\n\n- **Specific density:** ...


## Answering cost and time per question

OpenAI models are priced per token; Qwen per second of model working time (llama-server's prompt + generation time) at `E.GPU_PRICE_PER_HOUR`. RunPod queueing, cold starts and job overhead are shown (`runpod_exec_s`, `runpod_delay_s`) but not charged. `wall_s` is the question's end-to-end time as the evaluation saw it.

In [8]:
E.answering_cost((RUN,)).T

run,test-F
answers,510
calls/q,3.0
prompt_tokens/q,2428
completion_tokens/q,411
model_s/q,None
runpod_exec_s/q,None
runpod_delay_s/q,None
wall_s/q (median),7.4
"cost per 1,000 questions ($)",0.98


## Judge spend (Anthropic)

In [9]:
spend = E.judge_spend((RUN,))
spend.groupby("model")[["calls", "input_tokens", "output_tokens", "cost_usd"]].sum()

,calls,input_tokens,output_tokens,cost_usd
model,,,,
claude-haiku-5-5,2762,5870164,776008,0.9751
claude-sonnet-5-5,1050,1865189,41712,4.1475
